In [ ]:
import random
import copy

class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action, heuristic_cost):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.heuristic_cost = heuristic_cost
        
    def get_tuple_floor_state(self):
        return tuple(tuple(row) for row in self.floor_state)
    
GOAL_STATE = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
ROW, COL = 3, 3

def floor_and_vacuumpos_initialize():
    floor = []
    vacuum_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    for i in range(ROW):
        floor.append([random.randint(0, 1) for _ in range(COL)])
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def static_initialize():
    floor = [[0, 1, 0],
             [1, 0, 1],
             [1, 0, 1]]
    vacuum_pos = (0, 0)
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    moves = []
    if vacuum_pos[0] > 0: moves.append("UP")
    if vacuum_pos[0] < ROW - 1: moves.append("DOWN")
    if vacuum_pos[1] > 0: moves.append("LEFT")
    if vacuum_pos[1] < COL - 1: moves.append("RIGHT")
    return moves

def get_n0_of_dirty_tiles(floor: list) -> int:
    number_of_dirty_tile = 0
    for row in floor:
        for tile in row:
            if tile == 1:
                number_of_dirty_tile += 1
    return number_of_dirty_tile

def min_manhattan_distance(floor, vacuum_pos) -> int:
    min_dist = float('inf')
    found_dirty = False
    for r in range(ROW):
        for c in range(COL):
            if floor[r][c] == 1:
                found_dirty = True
                dist = abs(r - vacuum_pos[0]) + abs(c - vacuum_pos[1])
                if dist < min_dist:
                    min_dist = dist
    return min_dist if found_dirty else 0

def get_heuristic_cost(floor, vacuum_pos) -> int:
    # Hàm heuristic cải tiến tránh Flat Plateau
    dirty_count = get_n0_of_dirty_tiles(floor)
    manhattan_dist = min_manhattan_distance(floor, vacuum_pos)
    return dirty_count + manhattan_dist

def apply_move(floor, vacuum_pos, move):
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos

def print_result(node, step, success=True):
    if success:
        print(f"\nĐã tìm thấy đáp án thành công tại Step {step}.")
    else:
        print(f"\nKhông tìm thấy đáp án, dừng ở Step {step}.")
    
    # Truy vết đường đi
    path = []
    curr = node
    while curr is not None:
        path.append(curr)
        curr = curr.parent
    path.reverse()
    
    for i, n in enumerate(path):
        action = f"Khởi tạo" if i == 0 else f"Đi [{n.birth_action}]"
        print(f"Bước {i}: {action} -> Vị trí: {n.position}, Số ô dơ: {get_n0_of_dirty_tiles(n.floor_state)}, Heuristic: {n.heuristic_cost}")
        for row in n.floor_state:
            print(row)
        print("-" * 20)


In [ ]:
def simple_hill_climbing(floor, vacuum_pos):
    current_node = Node(floor, vacuum_pos, None, None, get_heuristic_cost(floor, vacuum_pos))
    step = 0
    
    if floor == GOAL_STATE:
        print_result(current_node, step, success=True)
        return
        
    while True:
        can_continue = False
        step += 1
        moves = posible_moves(current_node.position)
        
        for move in moves:
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            child_heuristic_cost = get_heuristic_cost(temp_floor, temp_vacuum_pos)
            
            if child_heuristic_cost < current_node.heuristic_cost:
                child_node = Node(temp_floor, temp_vacuum_pos, current_node, move, child_heuristic_cost)
                current_node = child_node
                can_continue = True
                break
        
        if current_node.floor_state == GOAL_STATE:
            print_result(current_node, step, success=True)
            return
            
        if not can_continue:
            print_result(current_node, step, success=False)
            return



In [ ]:
def steepest_hill_climbing(floor, vacuum_pos):
    current_node = Node(floor, vacuum_pos, None, None, get_heuristic_cost(floor, vacuum_pos))
    step = 0
    
    if floor == GOAL_STATE:
        print_result(current_node, step, success=True)
        return

    while True:
        can_continue = False
        step += 1
        moves = posible_moves(current_node.position)
        children = []
        
        for move in moves:
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            child_heuristic_cost = get_heuristic_cost(temp_floor, temp_vacuum_pos)
            
            if child_heuristic_cost < current_node.heuristic_cost:
                child_node = Node(temp_floor, temp_vacuum_pos, current_node, move, child_heuristic_cost)
                children.append(child_node)
                
        if children:
            # Chọn nút con có heuristic nhỏ nhất (tốt nhất)
            current_node = min(children, key=lambda x: x.heuristic_cost)
            can_continue = True
            
        if current_node.floor_state == GOAL_STATE:
            print_result(current_node, step, success=True)
            return
            
        if not can_continue:
            print_result(current_node, step, success=False)
            return

In [ ]:

def stochastic_hill_climbing(floor, vacuum_pos):
    current_node = Node(floor, vacuum_pos, None, None, get_heuristic_cost(floor, vacuum_pos))
    step = 0
    
    if floor == GOAL_STATE:
        print_result(current_node, step, success=True)
        return

    while True:
        can_continue = False
        step += 1
        moves = posible_moves(current_node.position)
        children = []
        
        for move in moves:
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            child_heuristic_cost = get_heuristic_cost(temp_floor, temp_vacuum_pos)
            
            if child_heuristic_cost < current_node.heuristic_cost:
                child_node = Node(temp_floor, temp_vacuum_pos, current_node, move, child_heuristic_cost)
                children.append(child_node)
                
        if children:
            current_node = random.choice(children)
            can_continue = True
            
        if current_node.floor_state == GOAL_STATE:
            print_result(current_node, step, success=True)
            return
            
        if not can_continue:
            print_result(current_node, step, success=False)
            return

In [ ]:
def equal_steepest_hill_climbing(floor, vacuum_pos):
    current_node = Node(floor, vacuum_pos, None, None, get_heuristic_cost(floor, vacuum_pos))
    step = 0
    
    if floor == GOAL_STATE:
        print_result(current_node, step, success=True)
        return

    while True:
        can_continue = False
        step += 1
        moves = posible_moves(current_node.position)
        children = []
        
        for move in moves:
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            child_heuristic_cost = get_heuristic_cost(temp_floor, temp_vacuum_pos)
            
            if child_heuristic_cost <= current_node.heuristic_cost:
                child_node = Node(temp_floor, temp_vacuum_pos, current_node, move, child_heuristic_cost)
                children.append(child_node)
                
        if children:
            current_node = random.choice(children)
            can_continue = True
            
        if current_node.floor_state == GOAL_STATE:
            print_result(current_node, step, success=True)
            return
            
        if not can_continue:
            print_result(current_node, step, success=False)
            return

In [ ]:
def random_restart_hill_climbing(floor, vacuum_pos, max_restart: int):
    current_node = Node(floor, vacuum_pos, None, None, get_heuristic_cost(floor, vacuum_pos))
    step = 0
    loop_counter = 0
    
    if floor == GOAL_STATE:
        print_result(current_node, step, success=True)
        return
    
    while loop_counter < max_restart:
        step += 1
        moves = posible_moves(current_node.position)
        children = []
        
        for move in moves:
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move)
            child_heuristic_cost = get_heuristic_cost(temp_floor, temp_vacuum_pos)
            
            if child_heuristic_cost < current_node.heuristic_cost:
                child_node = Node(temp_floor, temp_vacuum_pos, current_node, move, child_heuristic_cost)
                children.append(child_node)
                
        if children:
            current_node = random.choice(children)
        else:
            loop_counter += 1
            continue
        
        if current_node.floor_state == GOAL_STATE:
            print_result(current_node, step, success=True)
            return
        
    print_result(current_node, step, success=False)
    return

In [ ]:

if __name__ == "__main__":
    # Khởi tạo một sàn chung để so sánh khách quan
    floor, vacuum_pos = static_initialize()
    
    steepest_hill_climbing(floor, vacuum_pos)